In [1]:
# =============================================================================
# CELL 1: CONFIGURATION
# =============================================================================

granularity = 'q'

START_DATE = '2023-01-01'
END_DATE = None

run_every_query = True

DATE_COL_MAP = {'q': 'book_date', 'm': 'book_date', 'w': 'app_date'}

NONKMX_LOBS = ['AN', 'FRN', 'STG', 'FLD', 'ENT']
POS_LOBS = ['AN', 'FRN', 'STG', 'FLD', 'ENT', 'KMX']

BASELINES = {
    'AN':  {'ltv': 1.94, 'new_recovery_unadjusted': 0.58, 'apr': 0.25},
    'FRN': {'ltv': 1.94, 'new_recovery_unadjusted': 0.54, 'apr': 0.25},
    'STG': {'ltv': 1.94, 'new_recovery_unadjusted': 0.60, 'apr': 0.25},
    'FLD': {'ltv': 1.45, 'new_recovery_unadjusted': 0.54, 'apr': 0.235},
    'ENT': {'ltv': 1.45, 'new_recovery_unadjusted': 0.55, 'apr': 0.235},
    'KMX': {'ltv': 1.59, 'new_recovery_unadjusted': 0.58, 'apr': 0.25},
}

MODEL_PARAMS = {
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
    'kmx_loss_scale': 0.067,
}

EXCEL_OUTPUT = '../output/pos_gl_attribution.xlsx'

In [2]:
# =============================================================================
# CELL 2: IMPORTS AND DERIVED CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import math
import os
import openpyxl

pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

date_col = DATE_COL_MAP[granularity]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()

PERIOD_FREQ_MAP = {'q': 'Q', 'm': 'M', 'w': 'W-SAT'}
period_freq = PERIOD_FREQ_MAP[granularity]

start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)

min_date_sql = f"'{START_DATE}'"

mean_unit_loss = MODEL_PARAMS['mean_unit_loss']
unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']

print(f"Granularity: {granularity}")
print(f"Date column: {date_col}")
print(f"Period range: {start_period} to {end_period}")
print(f"run_every_query: {run_every_query}")

Granularity: q
Date column: book_date
Period range: 2023Q1 to 2026Q3
run_every_query: True


In [3]:
# =============================================================================
# CELL 3: UTILITY FUNCTIONS
# =============================================================================

def run_sql(filename, sub_list=None, connection=None, filename_is_query=False, chunksize=200_000):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    conn = connection
    close_conn = False
    if conn is None:
        conn = pyodbc.connect("DSN=Redshift_prod_new")
        close_conn = True
    try:
        warnings.filterwarnings("ignore", category=UserWarning)
        chunks = []
        for chunk in pd.read_sql_query(sql=query, con=conn, chunksize=chunksize):
            chunks.append(chunk)
            print(f"  ... loaded {sum(len(c) for c in chunks):,} rows", end='\r')
        warnings.filterwarnings("default", category=UserWarning)
        df = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
        print(f"  ... loaded {len(df):,} rows total")
        return df
    finally:
        if close_conn:
            conn.close()


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(filename, pickle_name, sub_list=None, connection=None, force_refresh=False):
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(filename, sub_list=sub_list, connection=connection)
        store_pickle(df, pickle_name)
        return df
    return get_pickle(pickle_name)


def weighted_average_and_sum(group, metrics):
    if isinstance(metrics, str):
        weighted_avg = (group[metrics] * group.amt_financed).sum() / group.amt_financed.sum()
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        weighted_avg = (group[metric] * group.amt_financed).sum() / group.amt_financed.sum()
        result_dict[metric] = weighted_avg
    return pd.Series(result_dict)


def format_vintage(period_series):
    if len(period_series) == 0:
        return period_series.astype(str)
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)

In [4]:
# =============================================================================
# CELL 4: DATA FETCH
# =============================================================================

os.makedirs('cache', exist_ok=True)
force = run_every_query

need_conn = force or not all(
    os.path.exists(p) for p in ('../../cache/ula_v1.pkl', '../../cache/dla_v1.pkl', '../../cache/new_recovery_v1.pkl')
)

if need_conn:
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        ula_df_total = cached_sql(
            '../queries/vintage_level_ula_query.txt', '../../cache/ula_v1.pkl',
            sub_list=[('{min_book_date}', min_date_sql)],
            connection=conn, force_refresh=force,
        )
        print('ULA ready')
        dla_df = cached_sql(
            '../queries/new_dll_query.txt', '../../cache/dla_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('DLA ready')
        new_recovery = cached_sql(
            '../queries/new_recovery_queryt.txt', '../../cache/new_recovery_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('New recovery ready')
else:
    ula_df_total = get_pickle('../../cache/ula_v1.pkl')
    dla_df = get_pickle('../../cache/dla_v1.pkl')
    new_recovery = get_pickle('../../cache/new_recovery_v1.pkl')
    print('ULA, DLA, New recovery loaded from cache')

print(f"ULA records: {len(ula_df_total):,}")
print("[PROGRESS] Data Fetch Complete")

  ... loaded 7,719,851 rows total
ULA ready
  ... loaded 134,149 rows total
DLA ready
  ... loaded 1,209,643 rows total
New recovery ready
ULA records: 7,719,851
[PROGRESS] Data Fetch Complete


In [5]:
# =============================================================================
# CELL 5: DATA PREPARATION (ALL POS LOBS)
# =============================================================================

ula_df_total = ula_df_total[ula_df_total.lob != 'Core']

ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

for df in [ula_df_total, new_recovery]:
    df['quarter'] = pd.to_datetime(df[date_col]).dt.to_period('Q')
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')
    df['week'] = pd.to_datetime(df[date_col]).dt.to_period('W-SAT')
    period_key = {'q': 'quarter', 'm': 'month', 'w': 'week'}
    df['period'] = df[period_key[granularity]]

for df in [ula_df_total, new_recovery]:
    mask = (df['period'] >= start_period) & (df['period'] <= end_period)
    df.drop(df[~mask].index, inplace=True)

for df in [ula_df_total, new_recovery]:
    df['book_week'] = df['book_week'].astype(str)
    df['app_week'] = df['app_week'].astype(str)

date_col_str = f'{date_col}_str'
ula_df_total[date_col_str] = ula_df_total[date_col].astype(str)

# --- Common preparation ---
ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

# --- DLA Merge (for non-KMX pricing scalar) ---
dla_df = dla_df.rename(columns={'valid_vintage': 'book_vintage'})
dla_explicit = dla_df[dla_df['book_vintage'] != 'current']
dla_current = dla_df[dla_df['book_vintage'] == 'current'].drop(columns='book_vintage')
last_explicit_vintage = dla_explicit['book_vintage'].max()
ula_df_total = ula_df_total.merge(dla_explicit, how='left', on=['dealer_number', 'book_vintage'])
new_and_missing = ula_df_total['pricing_scalar'].isna() & (ula_df_total['book_vintage'] > last_explicit_vintage)
fallback = ula_df_total.loc[new_and_missing, ['dealer_number']].merge(dla_current, on='dealer_number', how='left')
for col in ['dll_edition', 'loss_ratio', 'dealer_level', 'pricing_scalar']:
    ula_df_total.loc[new_and_missing, col] = fallback[col].values
ula_df_total['pricing_scalar'] = ula_df_total['pricing_scalar'].fillna(1)
ula_df_total.loc[ula_df_total.frni_flag == 1, 'pricing_scalar'] *= 1.05
ula_df_total.loc[ula_df_total.frni_flag == 0, 'pricing_scalar'] *= 0.95

# --- Driver Flag ---
warnings.filterwarnings('ignore', category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('not provided')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings('default', category=UserWarning)

# --- NA Handling ---
ula_df_total = ula_df_total.dropna(subset=['sale_price', 'cd_model_score', 'pti', 'lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# --- Filters ---
ula_df_total['bbltv'] = ula_df_total.amt_financed / ula_df_total.bbvalue.replace(0, np.nan)
ula_df_total = ula_df_total[ula_df_total.amt_financed <= 75000]
ula_df_total = ula_df_total[ula_df_total.pti <= 0.6]
ula_df_total = ula_df_total[
    (ula_df_total.lob == 'MCY') |
    (ula_df_total.bbltv <= 10.0) |
    (ula_df_total.bbvalue.isna()) |
    (ula_df_total.bbvalue == 0)
]
ula_df_total = ula_df_total[ula_df_total.total_income <= 200000]

# --- NonKMX-specific Flags ---
ula_df_total['ent_fld_flag'] = (ula_df_total.lob == 'ENT') | (ula_df_total.lob == 'FLD')
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500) & (ula_df_total.lob != 'MCY')
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY')
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.3) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['toyho_flag'] = ula_df_total.make.isin({'TOY', 'HON'})
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col_str] >= '2022-07-01') & (ula_df_total[date_col_str] < '2025-01-01')
ula_df_total['mcy_low_mileage_flag'] = (ula_df_total.lob == 'MCY') & (ula_df_total.cd_model_score > 140) & (ula_df_total.mileage <= 20000) & (ula_df_total.vehicle_age <= 10)
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['pricing_change_flag'] = ula_df_total[date_col_str] >= '2024-10-01'
ula_df_total['ent_flag'] = (ula_df_total.lob == 'ENT')
ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col_str] >= '2023-05-01'
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & ula_df_total.lob.isin({'AN', 'FLD', 'FRN', 'STG'}) & (ula_df_total.cb_flag)
ula_df_total['student_loan_flag'] = np.where(ula_df_total.student_loan_flag == 1, 1, 0)

# --- KMX-specific Flags ---
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['null_fico_null_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & ((ula_df_total.vantage_score < 300) | (ula_df_total.vantage_score > 850))
ula_df_total['high_sales_price_flag'] = ula_df_total.sale_price > 30000
ula_df_total['kmx_npc_flag'] = ula_df_total.kmx_npc_flag == 1
ula_df_total['high_pti_npc'] = ula_df_total.pti > 0.2
ula_df_total['job_time_flag'] = ula_df_total.employed_months < 6
ula_df_total['low_fico_flag'] = (ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 450)
ula_df_total['high_model_score_flag'] = ula_df_total.cd_model_score >= 146
ula_df_total['low_vantage_flag'] = (ula_df_total.fico_score < 300) & (ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450)
ula_df_total['louisiana_flag'] = ula_df_total.state == 'LA'
ula_df_total['txca_flag'] = ula_df_total.state.isin(['TX', 'CA', 'FL', 'CO'])
ula_df_total['illinois_flag'] = ula_df_total.state == 'IL'
ula_df_total['mississippi_flag'] = ula_df_total.state == 'MS'
ula_df_total['georgia_flag'] = ula_df_total.state == 'GA'
ula_df_total['normal_pti_flag'] = ula_df_total.pti <= 0.2
ula_df_total['high_pti_tier_1_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.pti <= 0.25)
ula_df_total['high_pti_tier_2_flag'] = (ula_df_total.pti > 0.25) & (ula_df_total.pti <= 0.35)
ula_df_total['high_pti_tier_3_flag'] = ula_df_total.pti > 0.35
ula_df_total['existing_dq_flag'] = ula_df_total.existing_dq_count > 0
ula_df_total['kmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines >= 0.14
ula_df_total['low_bureau_flag'] = ((ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 475)) | ((ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450))
ula_df_total['soft_pull_flag'] = ula_df_total.pull_type.isin(['softpull', 'prequal'])
ula_df_total['cd_perc_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1)
ula_df_total['open_tl_flag'] = ula_df_total.open_tl == 0
ula_df_total['narrowed_soft_pull_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1) & ula_df_total.soft_pull_flag & (~ula_df_total.job_time_flag)

# --- Deduplicate driver flags ---
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

# --- Filter to POS LOBs and apply MTN 4.1 transform for KMX ---
ula_df_total = ula_df_total[ula_df_total.lob.isin(POS_LOBS)].copy()
is_mtn41 = ula_df_total.mtn_model == 4.1
ula_df_total.loc[is_mtn41, 'cd_model_score'] = (
    142 + (ula_df_total.loc[is_mtn41, 'cd_model_score'] - 142) * 1.5
)

# --- Vintage assignment ---
ula_df_total['vintage'] = format_vintage(ula_df_total['period'])
new_recovery['vintage'] = format_vintage(new_recovery['period'])

# --- Model score aggregation per LOB ---
ms_df = ula_df_total.groupby(['period', 'lob']).apply(
    weighted_average_and_sum, 'cd_model_score', include_groups=False
).reset_index()
ms_df = ms_df.rename(columns={'cd_model_score': 'model_score'})
ms_df['period'] = format_vintage(ms_df['period'])

all_vintages = sorted(ula_df_total['vintage'].unique())

print(f"ULA after filters (POS LOBs): {len(ula_df_total):,}")
print(f"LOBs: {sorted(ula_df_total.lob.unique())}")
print(f"Vintages: {len(all_vintages)}")

ULA after filters (POS LOBs): 458,208
LOBs: ['AN', 'ENT', 'FLD', 'FRN', 'KMX', 'STG']
Vintages: 15


In [6]:
# =============================================================================
# CELL 6: MULTIPLIER & SCORING FUNCTIONS
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df):
    ula_df['loss_multiplier'] = 1.0
    ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag
    ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)
    ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)
    ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag
    ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)
    ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                               - 0.1 * ula_df.car_make_benefit_flag
                               - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)
    ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                               + 0.013 * ula_df.pricing_change_flag)
    ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag
    ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag
    ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
        - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
        * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date
    ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag
    ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag
    ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag
    ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag
    ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)
    ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag
    ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)
    ula_df.loss_multiplier *= ula_df.pricing_scalar
    ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)
    return ula_df


def get_ula_multiplier_nonkmx_diag(ula_df):
    steps = {}
    def _record(label, df):
        steps[label] = df.loss_multiplier.mean()

    ula_df['loss_multiplier'] = 1.0
    _record('00_initial', ula_df)
    ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag
    _record('01_prev_aca_chargeoff', ula_df)
    ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)
    _record('02_small_amt_financed', ula_df)
    ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)
    _record('03_zero_cash_down', ula_df)
    ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag
    _record('04_high_mileage', ula_df)
    ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)
    _record('05_high_pti', ula_df)
    ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                               - 0.1 * ula_df.car_make_benefit_flag
                               - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)
    _record('06_car_make', ula_df)
    ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                               + 0.013 * ula_df.pricing_change_flag)
    _record('07_theft_risk', ula_df)
    ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag
    _record('08_mcy_low_mileage', ula_df)
    ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag
    _record('09_weekend_weekday', ula_df)
    ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
        - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
        * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date
    _record('10_student_loans', ula_df)
    ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag
    _record('11_low_pti', ula_df)
    ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag
    _record('12_chime', ula_df)
    ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag
    _record('13_employment_type', ula_df)
    ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag
    _record('14_auth_tradelines', ula_df)
    ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)
    _record('15_fraud', ula_df)
    ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag
    _record('16_driver_flag', ula_df)
    ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)
    ula_df.loss_multiplier *= ula_df.pricing_scalar
    _record('17_pricing_scalar', ula_df)
    ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)
    _record('18_final', ula_df)

    flags = {
        'prev_co_flag': ula_df.prev_co_flag.mean(),
        'small_amt_financed_flag': ula_df.small_amt_financed_flag.mean(),
        'pricing_change_flag': ula_df.pricing_change_flag.mean(),
        'zero_cash_down_flag': ula_df.zero_cash_down_flag.mean(),
        'high_mileage_vehicle_flag': ula_df.high_mileage_vehicle_flag.mean(),
        'high_pti_flag': ula_df.high_pti_flag.mean(),
        'car_make_penalty_flag': ula_df.car_make_penalty_flag.mean(),
        'car_make_benefit_flag': ula_df.car_make_benefit_flag.mean(),
        'theft_risk_flag': ula_df.theft_risk_flag.mean(),
        'mcy_low_mileage_flag': ula_df.mcy_low_mileage_flag.mean(),
        'weekend_flag': ula_df.weekend_flag.mean(),
        'weekday_flag': ula_df.weekday_flag.mean(),
        'student_loan_flag': ula_df.student_loan_flag.mean(),
        'student_loans_cutoff_date': ula_df.student_loans_cutoff_date.mean(),
        'low_pti_flag': ula_df.low_pti_flag.mean(),
        'nonkmx_chime_flag': ula_df.nonkmx_chime_flag.mean(),
        'seasonal_employment_flag': ula_df.seasonal_employment_flag.mean(),
        'waiter_employment_flag': ula_df.waiter_employment_flag.mean(),
        'nonkmx_auth_tradelines_flag': ula_df.nonkmx_auth_tradelines_flag.mean(),
        'fraud_adjustment_mean': ula_df.fraud_adjustment.mean(),
        'driver_flag': ula_df.driver_flag.mean(),
        'pricing_scalar_mean': ula_df.pricing_scalar.mean(),
        'toyho_flag': ula_df.toyho_flag.mean(),
    }
    return ula_df, pd.Series(steps), pd.Series(flags)


def get_ula_multiplier_kmx(ula_df):
    loss_scale = MODEL_PARAMS['kmx_loss_scale']
    ula_df['loss_multiplier'] = 1.0
    ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag) + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag)
    ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag) + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag)
    ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 1 + 0 * ula_df.normal_pti_flag + 0.05 * ula_df.high_pti_tier_1_flag + 0.1 * ula_df.high_pti_tier_2_flag + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag
    ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] /= (1 + loss_scale)
    ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 1 + (-0.01 + 0.06 * ula_df.secured_credit_flag) * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag)
    ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag
    ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= (1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025 + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108 + ula_df.secured_credit_flag * 0.295)))
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + (ula_df.fraud_adjustment - 1)
    clipped = np.clip(ula_df.loc[(ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag), 'loss_multiplier'], 0.8, 1.35 / (1 + loss_scale))
    ula_df.loc[(ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag), 'loss_multiplier'] = clipped
    ula_df.loc[ula_df.mtn_model.isin([3.0]),'loss_multiplier'] *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age
    ula_df.loc[ula_df.kmx_npc_flag, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.high_pti_npc
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 0.96 + 0.14 * ula_df.student_loan_flag
    ula_df.loc[ula_df.mtn_model.isin([4.1]),'loss_multiplier'] *= 0.98 + 0.22 * ula_df.high_sales_price_flag
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 1 + 0.15 * ula_df.driver_flag
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 1 + 0.35 * ula_df.louisiana_flag
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 1 + 0.1 * ula_df.georgia_flag
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score > 140),'loss_multiplier'] *= 1 - 0.1 * ula_df.txca_flag
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score <= 140) & (ula_df.cd_model_score >= 135),'loss_multiplier'] *= 1 - 0.05 * ula_df.txca_flag
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 1 + 0.25 * ula_df.illinois_flag
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 1 + 0.2 * ula_df.mississippi_flag
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 0.96 + (0.01 * ula_df.soft_pull_flag - 0.18 * ula_df.chime_flag * ula_df.soft_pull_flag) + 0.46 * ula_df.chime_flag
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]),'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag
    ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]),'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag
    ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ula_df.soft_pull_flag,'loss_multiplier'] *= 1.1 * (0.99 + 0.11 * ula_df.low_bureau_flag) * (0.978 + 0.172 * ula_df.open_tl_flag)
    ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ~ula_df.soft_pull_flag,'loss_multiplier'] *= (0.98 + 0.22 * ula_df.low_bureau_flag) * (0.945 + 0.405 * ula_df.open_tl_flag) / np.maximum(ula_df.cd_perc_flag * ula_df.open_tl_flag * 1.2, 1)
    ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.97 + 0.15 * ula_df.cd_perc_flag
    ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.954 + 0.346 * ula_df.cd_perc_flag
    ula_df.loc[(ula_df.mtn_model == 4.1) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.05 * ula_df.cd_perc_flag
    ula_df.loc[(ula_df.mtn_model == 4.1) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.10 * ula_df.cd_perc_flag
    ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] /= 1.1
    ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] = np.clip(ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'], 0.75, 1.4)
    return ula_df


def get_ula_multiplier_kmx_diag(ula_df):
    loss_scale = MODEL_PARAMS['kmx_loss_scale']
    steps = {}
    def _record(label, df):
        steps[label] = df.loss_multiplier.mean()

    ula_df['loss_multiplier'] = 1.0
    _record('00_initial', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag) + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag)
    _record('02_low_fico_3.0', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag) + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag)
    _record('03_low_vantage_3.0', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 1 + 0.05 * ula_df.high_pti_tier_1_flag + 0.1 * ula_df.high_pti_tier_2_flag + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag
    _record('04_high_pti_3.0', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] /= (1 + loss_scale)
    _record('07_loss_scale_div_3.0', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 1 + (-0.01 + 0.06 * ula_df.secured_credit_flag) * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag)
    _record('08_secured_credit_3.0', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag
    _record('09_auth_tradelines_3.0', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025 + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108 + ula_df.secured_credit_flag * 0.295)))
    _record('11_soft_pull_3.0', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + (ula_df.fraud_adjustment - 1)
    _record('12_fraud_all', ula_df)
    clipped = np.clip(ula_df.loc[(ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag), 'loss_multiplier'], 0.8, 1.35 / (1 + loss_scale))
    ula_df.loc[(ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag), 'loss_multiplier'] = clipped
    _record('13_clip_3.0', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age
    _record('14_vehicle_age_3.0', ula_df)
    ula_df.loc[ula_df.kmx_npc_flag, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.high_pti_npc
    _record('15_npc_all', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.96 + 0.14 * ula_df.student_loan_flag
    _record('16_student_loans_all', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([4.1]), 'loss_multiplier'] *= 0.98 + 0.22 * ula_df.high_sales_price_flag
    _record('17_high_sales_price_4.1', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.15 * ula_df.driver_flag
    _record('18_driver_flag_all', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.35 * ula_df.louisiana_flag
    _record('19_louisiana_all', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.georgia_flag
    _record('20_georgia_all', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score > 140), 'loss_multiplier'] *= 1 - 0.1 * ula_df.txca_flag
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score <= 140) & (ula_df.cd_model_score >= 135), 'loss_multiplier'] *= 1 - 0.05 * ula_df.txca_flag
    _record('21_txca_all', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.25 * ula_df.illinois_flag
    _record('22_illinois_all', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.2 * ula_df.mississippi_flag
    _record('23_mississippi_all', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.96 + (0.01 * ula_df.soft_pull_flag - 0.18 * ula_df.chime_flag * ula_df.soft_pull_flag) + 0.46 * ula_df.chime_flag
    _record('24_secured_credit_3.1+', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag
    _record('25_job_time_3.1+', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag
    _record('26_existing_dq_3.1+', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag
    _record('27_employment_3.1+', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag
    _record('28_auth_tradelines_3.1+', ula_df)
    mtn31_mask = ula_df.mtn_model.isin([3.1, 3.2, 4.1])
    sp_mask = mtn31_mask & ula_df.soft_pull_flag
    hp_mask = mtn31_mask & ~ula_df.soft_pull_flag
    ula_df.loc[sp_mask, 'loss_multiplier'] *= 1.1 * (0.99 + 0.11 * ula_df.low_bureau_flag) * (0.978 + 0.172 * ula_df.open_tl_flag)
    _record('29_sp_bureau_3.1+', ula_df)
    ula_df.loc[hp_mask, 'loss_multiplier'] *= (0.98 + 0.22 * ula_df.low_bureau_flag) * (0.945 + 0.405 * ula_df.open_tl_flag) / np.maximum(ula_df.cd_perc_flag * ula_df.open_tl_flag * 1.2, 1)
    _record('30_hp_bureau_3.1+', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.97 + 0.15 * ula_df.cd_perc_flag
    ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.954 + 0.346 * ula_df.cd_perc_flag
    ula_df.loc[(ula_df.mtn_model == 4.1) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.05 * ula_df.cd_perc_flag
    ula_df.loc[(ula_df.mtn_model == 4.1) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.10 * ula_df.cd_perc_flag
    _record('31_cd_perc_3.1+', ula_df)
    ula_df.loc[mtn31_mask, 'loss_multiplier'] /= 1.1
    _record('32_blanket_3.1+', ula_df)
    ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] = np.clip(ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'], 0.75, 1.4)
    _record('33_final_clip_3.1+', ula_df)

    flags = {
        'job_time_flag': ula_df.job_time_flag.mean(),
        'low_fico_flag': ula_df.low_fico_flag.mean(),
        'high_model_score_flag': ula_df.high_model_score_flag.mean(),
        'low_vantage_flag': ula_df.low_vantage_flag.mean(),
        'normal_pti_flag': ula_df.normal_pti_flag.mean(),
        'high_pti_tier_1_flag': ula_df.high_pti_tier_1_flag.mean(),
        'high_pti_tier_2_flag': ula_df.high_pti_tier_2_flag.mean(),
        'high_pti_tier_3_flag': ula_df.high_pti_tier_3_flag.mean(),
        'existing_dq_flag': ula_df.existing_dq_flag.mean(),
        'seasonal_employment_flag': ula_df.seasonal_employment_flag.mean(),
        'secured_credit_flag': ula_df.secured_credit_flag.mean(),
        'kmx_auth_tradelines_flag': ula_df.kmx_auth_tradelines_flag.mean(),
        'null_fico_w_vantage_flag': ula_df.null_fico_w_vantage_flag.mean(),
        'null_fico_null_vantage_flag': ula_df.null_fico_null_vantage_flag.mean(),
        'soft_pull_flag': ula_df.soft_pull_flag.mean(),
        'narrowed_soft_pull_flag': ula_df.narrowed_soft_pull_flag.mean(),
        'fraud_adjustment_mean': ula_df.fraud_adjustment.mean(),
        'continuous_vehicle_age_mean': ula_df.continuous_vehicle_age.mean(),
        'kmx_npc_flag': ula_df.kmx_npc_flag.mean(),
        'high_pti_npc': ula_df.high_pti_npc.mean(),
        'student_loan_flag': ula_df.student_loan_flag.mean(),
        'high_sales_price_flag': ula_df.high_sales_price_flag.mean(),
        'driver_flag': ula_df.driver_flag.mean(),
        'louisiana_flag': ula_df.louisiana_flag.mean(),
        'georgia_flag': ula_df.georgia_flag.mean(),
        'txca_flag': ula_df.txca_flag.mean(),
        'illinois_flag': ula_df.illinois_flag.mean(),
        'mississippi_flag': ula_df.mississippi_flag.mean(),
        'chime_flag': ula_df.chime_flag.mean(),
        'low_bureau_flag': ula_df.low_bureau_flag.mean(),
        'cd_perc_flag': ula_df.cd_perc_flag.mean(),
        'open_tl_flag': ula_df.open_tl_flag.mean(),
        'mtn_model_3.0_pct': (ula_df.mtn_model == 3.0).mean(),
        'mtn_model_3.1_pct': (ula_df.mtn_model == 3.1).mean(),
        'mtn_model_3.2_pct': (ula_df.mtn_model == 3.2).mean(),
        'mtn_model_4.1_pct': (ula_df.mtn_model == 4.1).mean(),
        'toyho_flag': ula_df.toyho_flag.mean(),
    }
    return ula_df, pd.Series(steps), pd.Series(flags)


def get_ragu_score_for_lob(vintage, lob, ula_source, new_recovery_source, ms_source, baseline_config):
    baseline_ltv = baseline_config['ltv']
    baseline_recovery = baseline_config['new_recovery_unadjusted']
    baseline_apr = baseline_config['apr']
    ltv_mult = 17 / 0.65 if lob == 'KMX' else 17
    apr_mult = 0.7 / 0.65 if lob == 'KMX' else 0.7

    ula_df = ula_source[(ula_source.vintage == vintage) & (ula_source.lob == lob)].copy()
    if len(ula_df) == 0:
        return None

    if lob == 'KMX':
        ula_df = get_ula_multiplier_kmx(ula_df)
    else:
        ula_df = get_ula_multiplier_nonkmx(ula_df)

    ula_df = ula_df[['account_number', date_col, 'bbvalue', 'sale_price', 'amt_financed', 'lob_or_bucket', 'lob', 'loss_multiplier', 'apr']]
    nr = new_recovery_source[['account_number', 'new_recovery_multiplier']].drop_duplicates(subset='account_number', keep='first')
    mix_df = ula_df.merge(nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
                          on='account_number', how='left').drop_duplicates(subset='account_number', keep='first')

    mix_df['ltv'] = mix_df.amt_financed / mix_df.bbvalue

    full_pop_loss = mix_df.groupby('lob').apply(
        weighted_average_and_sum, ['loss_multiplier'], include_groups=False
    )
    full_pop_loss = full_pop_loss.rename(columns={'amt_financed': 'amt_financed_full'})

    bb_populated_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0)]
    if len(bb_populated_df) == 0:
        return None

    bb_metrics = bb_populated_df.groupby('lob').apply(
        weighted_average_and_sum, ['ltv', 'bbvalue', 'apr'], include_groups=False
    )

    full_pop_metrics = full_pop_loss.join(bb_metrics.drop(columns='amt_financed'))
    full_pop_metrics['amt_financed'] = full_pop_loss['amt_financed_full']
    full_pop_metrics = full_pop_metrics.drop(columns='amt_financed_full')

    recovery_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0) & mix_df['recovery_multiplier'].notna()].copy()
    recovery_df['recovery_unadjusted_multiplier'] = recovery_df['recovery_multiplier']
    recovery_metrics = recovery_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['recovery_unadjusted_multiplier'],
        include_groups=False
    ).drop(columns='amt_financed')

    grouped = full_pop_metrics.join(recovery_metrics)

    vintage_ms = ms_source[ms_source['period'] == vintage].copy()

    idx_name = grouped.index.name
    if isinstance(idx_name, str) and idx_name in grouped.columns:
        grouped = grouped.reset_index(drop=True)
    else:
        grouped = grouped.reset_index()

    full_df = grouped.merge(vintage_ms, on='lob')
    if len(full_df) == 0:
        return None

    full_df['est_unit_loss'] = mean_unit_loss
    full_df['unit_loss_score'] = full_df.model_score + (1 - full_df.loss_multiplier) * mean_unit_loss / unit_loss_to_model_score
    full_df = full_df.set_index('lob')
    full_df['ms_original'] = full_df.model_score.copy()
    full_df['baselined_unadjusted_recovery'] = full_df.recovery_unadjusted_multiplier / baseline_recovery
    full_df['gross_loss_impact'] = full_df['unit_loss_score'] - full_df['ms_original']
    full_df['recovery_impact'] = (full_df['unit_loss_score'] * full_df['est_unit_loss']
        * full_df['recovery_unadjusted_multiplier'] * (full_df['baselined_unadjusted_recovery'] - 1))
    full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * ltv_mult
    full_df['apr_impact'] = (baseline_apr - full_df['apr']) / 0.01 * apr_mult
    full_df['ragu_score'] = (
        (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score
        + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier
        * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
        + full_df['ltv_impact'] + full_df['apr_impact'])
    full_df['vintage'] = vintage
    return full_df

In [7]:
# =============================================================================
# CELL 7: RAGU SCORING + PER-LOB ATTRIBUTION
# =============================================================================

NONKMX_STEP_LABELS = {
    '01_prev_aca_chargeoff': 'Previous ACA Chargeoff (nonKMX)',
    '02_small_amt_financed': 'Small Amount Financed (nonKMX)',
    '03_zero_cash_down':     'Zero Cash Down (nonKMX)',
    '04_high_mileage':       'High Mileage Vehicle (nonKMX)',
    '05_high_pti':           'High PTI (nonKMX)',
    '06_car_make':           'Car Make (nonKMX)',
    '07_theft_risk':         'Theft Risk (nonKMX)',
    '08_mcy_low_mileage':    'MCY Low Mileage (nonKMX)',
    '09_weekend_weekday':    'Weekend / Weekday (nonKMX)',
    '10_student_loans':      'Student Loans (nonKMX)',
    '11_low_pti':            'Low PTI (nonKMX)',
    '12_chime':              'Chime (nonKMX)',
    '13_employment_type':    'Employment Type (nonKMX)',
    '14_auth_tradelines':    'Auth Tradelines (nonKMX)',
    '15_fraud':              'Fraud (nonKMX)',
    '16_driver_flag':        'Driver Flag (nonKMX)',
    '17_pricing_scalar':     'Dealer Level / Pricing Scalar (nonKMX)',
    '18_final':              'Final Clip (nonKMX)',
}

KMX_STEP_LABELS = {
    '02_low_fico_3.0':         'Low FICO (KMX 3.0)',
    '03_low_vantage_3.0':      'Low Vantage (KMX 3.0)',
    '04_high_pti_3.0':         'High PTI (KMX 3.0)',
    '07_loss_scale_div_3.0':   'Loss Scale Div (KMX 3.0)',
    '08_secured_credit_3.0':   'Secured Credit (KMX 3.0)',
    '09_auth_tradelines_3.0':  'Auth Tradelines (KMX 3.0)',
    '11_soft_pull_3.0':        'Soft Pull (KMX 3.0)',
    '12_fraud_all':            'Fraud (KMX)',
    '13_clip_3.0':             'Clip (KMX 3.0)',
    '14_vehicle_age_3.0':      'Vehicle Age (KMX)',
    '15_npc_all':              'NPC (KMX)',
    '16_student_loans_all':    'Student Loans (KMX)',
    '17_high_sales_price_4.1': 'High Sales Price (KMX 4.1)',
    '18_driver_flag_all':      'Driver Flag (KMX)',
    '19_louisiana_all':        'Louisiana (KMX)',
    '20_georgia_all':          'Georgia (KMX)',
    '21_txca_all':             'TX / CA / FL / CO (KMX)',
    '22_illinois_all':         'Illinois (KMX)',
    '23_mississippi_all':      'Mississippi (KMX)',
    '24_secured_credit_3.1+':  'Chime / Secured Credit (KMX)',
    '25_job_time_3.1+':        'Job Time (KMX 3.1+)',
    '26_existing_dq_3.1+':     'Existing DQ (KMX 3.1+)',
    '27_employment_3.1+':      'Employment Type (KMX)',
    '28_auth_tradelines_3.1+': 'Auth Tradelines (KMX 3.1+)',
    '29_sp_bureau_3.1+':       'Low Bureau SP (KMX 3.1+)',
    '30_hp_bureau_3.1+':       'Low Bureau HP (KMX 3.1+)',
    '31_cd_perc_3.1+':         'CD Perc (KMX 3.1+)',
    '32_blanket_3.1+':         'Blanket Adjustment (KMX 3.1+)',
    '33_final_clip_3.1+':      'Clip (KMX 3.1+)',
}


def log_ratio_attribution(steps_dict, gross_loss_impact):
    """Compute per-step attribution using log-ratio decomposition.
    Returns dict {readable_label: attributed_impact}."""
    step_keys = list(steps_dict.keys())
    adjustment_keys = [k for k in step_keys if k != '00_initial']
    cumulative = [steps_dict[k] for k in step_keys]

    ratios = []
    for i in range(1, len(cumulative)):
        prev, curr = cumulative[i - 1], cumulative[i]
        ratios.append(curr / prev if prev and prev != 0 else 1.0)

    final_mult = cumulative[-1]
    diag_gli = 25.0 * (1.0 - final_mult) if final_mult and not pd.isna(final_mult) else 0.0
    log_final = math.log(final_mult) if final_mult and final_mult > 0 and abs(final_mult - 1.0) > 1e-12 else None

    attributed = {}
    if log_final is None:
        for k in adjustment_keys:
            attributed[k] = 0.0
    else:
        for k, r in zip(adjustment_keys, ratios):
            lr = math.log(r) if r and r > 0 else 0.0
            attributed[k] = (lr / log_final) * diag_gli

    pop_weighting = gross_loss_impact - sum(attributed.values())
    return attributed, pop_weighting


# --- Score all LOBs and collect per-LOB attributions ---
ragu_gli_by_lob = {}      # {(lob, vintage): gross_loss_impact}
af_by_lob = {}            # {(lob, vintage): total_amt_financed}
attribution_by_lob = {}   # {(lob, vintage): {step_label: impact}}
pop_weighting_by_lob = {} # {(lob, vintage): residual}
flag_results_by_lob = {}  # {(lob, vintage): pd.Series of flag means}
record_counts_by_lob = {} # {(lob, vintage): int}

for lob in POS_LOBS:
    baseline_config = BASELINES[lob]
    ula_lob = ula_df_total[ula_df_total.lob == lob]

    for vintage in all_vintages:
        result = get_ragu_score_for_lob(vintage, lob, ula_df_total, new_recovery, ms_df, baseline_config)
        if result is None:
            continue

        gli = result['gross_loss_impact'].iloc[0]
        af = result['amt_financed_y'].iloc[0]
        ragu_gli_by_lob[(lob, vintage)] = gli
        af_by_lob[(lob, vintage)] = af

        ula_vintage = ula_lob[ula_lob.vintage == vintage].copy()
        if len(ula_vintage) == 0:
            continue

        record_counts_by_lob[(lob, vintage)] = len(ula_vintage)

        if lob == 'KMX':
            _, steps, flag_series = get_ula_multiplier_kmx_diag(ula_vintage)
            label_map = KMX_STEP_LABELS
        else:
            _, steps, flag_series = get_ula_multiplier_nonkmx_diag(ula_vintage)
            label_map = NONKMX_STEP_LABELS

        flag_results_by_lob[(lob, vintage)] = flag_series

        attr_raw, pop_wt = log_ratio_attribution(steps.to_dict(), gli)
        attr_labeled = {label_map.get(k, k): v for k, v in attr_raw.items()}
        attribution_by_lob[(lob, vintage)] = attr_labeled
        pop_weighting_by_lob[(lob, vintage)] = pop_wt

    print(f"{lob}: {sum(1 for k in ragu_gli_by_lob if k[0] == lob)} vintages scored")

print(f"\nAll POS LOBs processed.")

AN: 15 vintages scored
FRN: 15 vintages scored
STG: 15 vintages scored
FLD: 15 vintages scored
ENT: 15 vintages scored
KMX: 15 vintages scored

All POS LOBs processed.


In [8]:
# =============================================================================
# CELL 8: POS-LEVEL ATTRIBUTION ROLL-UP
# =============================================================================

POS_ROW_ORDER = [
    'Previous ACA Chargeoff (nonKMX)',
    'Small Amount Financed (nonKMX)',
    'Zero Cash Down (nonKMX)',
    'Vehicle Age (KMX)',
    'High Mileage Vehicle (nonKMX)',
    'High Sales Price (KMX 4.1)',
    'High PTI (KMX 3.0)',
    'High PTI (nonKMX)',
    'Low PTI (nonKMX)',
    'NPC (KMX)',
    'Low FICO (KMX 3.0)',
    'Low Vantage (KMX 3.0)',
    'Car Make (nonKMX)',
    'Theft Risk (nonKMX)',
    'Louisiana (KMX)',
    'Georgia (KMX)',
    'TX / CA / FL / CO (KMX)',
    'Illinois (KMX)',
    'Mississippi (KMX)',
    'MCY Low Mileage (nonKMX)',
    'Weekend / Weekday (nonKMX)',
    'Student Loans (KMX)',
    'Student Loans (nonKMX)',
    'Chime / Secured Credit (KMX)',
    'Chime (nonKMX)',
    'Secured Credit (KMX 3.0)',
    'Auth Tradelines (KMX 3.0)',
    'Auth Tradelines (KMX 3.1+)',
    'Auth Tradelines (nonKMX)',
    'Low Bureau SP (KMX 3.1+)',
    'Low Bureau HP (KMX 3.1+)',
    'CD Perc (KMX 3.1+)',
    'Soft Pull (KMX 3.0)',
    'Job Time (KMX 3.1+)',
    'Employment Type (KMX)',
    'Employment Type (nonKMX)',
    'Fraud (KMX)',
    'Fraud (nonKMX)',
    'Driver Flag (KMX)',
    'Driver Flag (nonKMX)',
    'Existing DQ (KMX 3.1+)',
    'Dealer Level / Pricing Scalar (nonKMX)',
    'Loss Scale Div (KMX 3.0)',
    'Blanket Adjustment (KMX 3.1+)',
    'Clip (KMX 3.0)',
    'Clip (KMX 3.1+)',
    'Final Clip (nonKMX)',
]


def build_pos_attribution():
    """Roll up per-LOB attributions to POS level using amt-financed weights."""
    pos_data = {}  # {vintage: {row_label: weighted_value}}

    for vintage in all_vintages:
        lob_attrs_for_vintage = []
        total_af = 0.0

        for lob in POS_LOBS:
            key = (lob, vintage)
            if key not in attribution_by_lob or key not in af_by_lob:
                continue
            af = af_by_lob[key]
            attr = attribution_by_lob[key]
            pw = pop_weighting_by_lob.get(key, 0.0)
            lob_attrs_for_vintage.append((af, attr, pw))
            total_af += af

        if total_af == 0 or not lob_attrs_for_vintage:
            continue

        all_labels = set()
        for _, attr, _ in lob_attrs_for_vintage:
            all_labels.update(attr.keys())

        vintage_row = {}
        for label in all_labels:
            weighted_sum = sum(af * attr.get(label, 0.0) for af, attr, _ in lob_attrs_for_vintage)
            vintage_row[label] = weighted_sum / total_af

        pw_weighted = sum(af * pw for af, _, pw in lob_attrs_for_vintage) / total_af
        vintage_row['--- Population Weighting / Mixing'] = pw_weighted

        pos_gli = sum(
            af_by_lob[(lob, vintage)] * ragu_gli_by_lob[(lob, vintage)]
            for lob in POS_LOBS
            if (lob, vintage) in ragu_gli_by_lob and (lob, vintage) in af_by_lob
        ) / total_af
        vintage_row['--- TOTAL (POS GLI)'] = pos_gli

        pos_data[vintage] = vintage_row

    return pos_data


pos_attribution_data = build_pos_attribution()

all_row_labels = set()
for vdata in pos_attribution_data.values():
    all_row_labels.update(vdata.keys())

ordered_rows = [r for r in POS_ROW_ORDER if r in all_row_labels]
extra_rows = sorted(all_row_labels - set(POS_ROW_ORDER) - {'--- Population Weighting / Mixing', '--- TOTAL (POS GLI)'})
final_row_order = ordered_rows + extra_rows + ['--- Population Weighting / Mixing', '--- TOTAL (POS GLI)']

pos_attribution_df = pd.DataFrame(
    {vintage: {row: pos_attribution_data[vintage].get(row, 0.0) for row in final_row_order}
     for vintage in sorted(pos_attribution_data.keys())},
    index=final_row_order
)

print("POS Attribution Table:")
display(pos_attribution_df)

POS Attribution Table:


,2023 Q1,2023 Q2,2023 Q3,2023 Q4,2024 Q1,2024 Q2,2024 Q3,2024 Q4,2025 Q1,2025 Q2,2025 Q3,2025 Q4,2026 Q1,2026 Q2,2026 Q3
Previous ACA Chargeoff (nonKMX),-0.001845,-0.001378,-0.001556,-0.002022,-0.002475,-0.002791,-0.003163,-0.003330,-0.004095,-0.005526,-0.003520,-0.003554,-0.006534,-0.008422,-0.007326
Small Amount Financed (nonKMX),-0.004724,-0.003990,-0.006030,-0.006367,-0.005963,-0.005927,-0.006322,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Zero Cash Down (nonKMX),-0.031488,-0.030575,-0.035462,-0.044522,-0.027315,-0.029747,-0.034237,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Vehicle Age (KMX),0.133477,0.135856,0.128732,0.201410,0.211032,0.369216,0.372233,0.278968,0.185777,0.153498,0.142472,0.110902,0.000000,0.000000,0.000000
High Mileage Vehicle (nonKMX),-0.013437,-0.016322,-0.015211,-0.018341,-0.018241,-0.012205,-0.006924,-0.013307,-0.008564,-0.004328,-0.004261,-0.003262,-0.004179,-0.003127,-0.002354
High Sales Price (KMX 4.1),0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000034,-0.012893,-0.025958,-0.047752
High PTI (KMX 3.0),-0.180065,-0.179481,-0.187712,-0.166125,-0.123072,-0.120241,-0.132426,-0.092145,-0.054154,-0.043391,-0.046206,-0.043283,0.000000,0.000000,0.000000
High PTI (nonKMX),-0.009945,-0.011234,-0.010517,-0.008101,-0.005485,-0.007224,-0.007817,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Low PTI (nonKMX),0.025160,0.026472,0.037366,0.044312,0.052945,0.076148,0.068362,0.069279,0.072366,0.071660,0.063387,0.056212,0.058733,0.063033,0.067395
NPC (KMX),-0.067225,-0.067395,-0.080124,-0.052062,-0.036546,-0.039766,-0.035930,-0.039532,-0.047141,-0.030930,-0.028067,-0.026259,-0.008970,-0.010705,-0.013968


In [9]:
# =============================================================================
# CELL 9: POS FLAG MEANS
# =============================================================================

POS_WIDE_FLAGS = ['student_loan_flag', 'fraud_adjustment_mean', 'driver_flag', 'seasonal_employment_flag', 'toyho_flag']
CHIME_NONKMX_KEY = 'nonkmx_chime_flag'
CHIME_KMX_KEY = 'chime_flag'

NONKMX_ONLY_FLAGS = [
    'prev_co_flag', 'small_amt_financed_flag', 'pricing_change_flag',
    'zero_cash_down_flag', 'high_mileage_vehicle_flag', 'high_pti_flag',
    'car_make_penalty_flag', 'car_make_benefit_flag', 'theft_risk_flag',
    'mcy_low_mileage_flag', 'weekend_flag', 'weekday_flag',
    'student_loans_cutoff_date', 'low_pti_flag', 'nonkmx_chime_flag',
    'waiter_employment_flag', 'nonkmx_auth_tradelines_flag', 'pricing_scalar_mean',
]

KMX_ONLY_FLAGS = [
    'low_fico_flag', 'low_vantage_flag', 'high_model_score_flag',
    'normal_pti_flag', 'high_pti_tier_1_flag', 'high_pti_tier_2_flag',
    'high_pti_tier_3_flag', 'secured_credit_flag', 'chime_flag',
    'soft_pull_flag', 'narrowed_soft_pull_flag', 'job_time_flag',
    'existing_dq_flag', 'kmx_auth_tradelines_flag',
    'null_fico_w_vantage_flag', 'null_fico_null_vantage_flag',
    'continuous_vehicle_age_mean', 'kmx_npc_flag', 'high_pti_npc',
    'high_sales_price_flag', 'louisiana_flag', 'georgia_flag', 'txca_flag',
    'illinois_flag', 'mississippi_flag', 'low_bureau_flag',
    'cd_perc_flag', 'open_tl_flag',
    'mtn_model_3.0_pct', 'mtn_model_3.1_pct', 'mtn_model_3.2_pct', 'mtn_model_4.1_pct',
]


def _weighted_flag_mean(flag_name, vintage, lob_list, flag_results, af_weights):
    """Compute amt_financed-weighted flag mean across LOBs for a single vintage."""
    data = []
    for lob in lob_list:
        key = (lob, vintage)
        if key in flag_results and key in af_weights:
            val = flag_results[key].get(flag_name, np.nan)
            if not pd.isna(val):
                data.append((val, af_weights[key]))
    if not data:
        return np.nan
    total_af = sum(d[1] for d in data)
    if total_af <= 0:
        return np.nan
    return sum(d[0] * d[1] for d in data) / total_af


def build_pos_flag_means(flag_results, af_weights, vintages, nonkmx_lobs, record_counts):
    blank = {'Flag': '', 'LOB_Applicability': '', **{v: '' for v in vintages}}
    rows = []

    # --- POS-WIDE FLAGS ---
    rows.append({'Flag': '--- POS-WIDE FLAGS ---', 'LOB_Applicability': '', **{v: '' for v in vintages}})

    for flag in POS_WIDE_FLAGS:
        row = {'Flag': flag, 'LOB_Applicability': 'POS (all LOBs)'}
        for vintage in vintages:
            row[vintage] = _weighted_flag_mean(flag, vintage, POS_LOBS, flag_results, af_weights)
        rows.append(row)

    # Chime (POS-wide): combine nonkmx_chime_flag + chime_flag
    row = {'Flag': 'Chime (POS-wide)', 'LOB_Applicability': 'POS (all LOBs)'}
    for vintage in vintages:
        data = []
        for lob in nonkmx_lobs:
            key = (lob, vintage)
            if key in flag_results and key in af_weights:
                val = flag_results[key].get(CHIME_NONKMX_KEY, np.nan)
                if not pd.isna(val):
                    data.append((val, af_weights[key]))
        kmx_key = ('KMX', vintage)
        if kmx_key in flag_results and kmx_key in af_weights:
            val = flag_results[kmx_key].get(CHIME_KMX_KEY, np.nan)
            if not pd.isna(val):
                data.append((val, af_weights[kmx_key]))
        if data:
            total_af = sum(d[1] for d in data)
            row[vintage] = sum(d[0] * d[1] for d in data) / total_af if total_af > 0 else np.nan
        else:
            row[vintage] = np.nan
    rows.append(row)

    rows.append(blank.copy())

    # --- NONKMX-ONLY FLAGS ---
    rows.append({'Flag': '--- NONKMX-ONLY FLAGS ---', 'LOB_Applicability': '', **{v: '' for v in vintages}})

    for flag in NONKMX_ONLY_FLAGS:
        row = {'Flag': flag, 'LOB_Applicability': 'NonKMX-only'}
        for vintage in vintages:
            row[vintage] = _weighted_flag_mean(flag, vintage, nonkmx_lobs, flag_results, af_weights)
        rows.append(row)

    rows.append(blank.copy())

    # --- KMX-ONLY FLAGS ---
    rows.append({'Flag': '--- KMX-ONLY FLAGS ---', 'LOB_Applicability': '', **{v: '' for v in vintages}})

    for flag in KMX_ONLY_FLAGS:
        row = {'Flag': flag, 'LOB_Applicability': 'KMX-only'}
        for vintage in vintages:
            kmx_key = ('KMX', vintage)
            if kmx_key in flag_results:
                row[vintage] = flag_results[kmx_key].get(flag, np.nan)
            else:
                row[vintage] = np.nan
        rows.append(row)

    rows.append(blank.copy())

    # --- RECORD COUNTS & AMT FINANCED ---
    rows.append({'Flag': '--- RECORD COUNTS ---', 'LOB_Applicability': '', **{v: '' for v in vintages}})

    row = {'Flag': 'NONKMX_RECORDS', 'LOB_Applicability': 'NonKMX-only'}
    for vintage in vintages:
        row[vintage] = sum(record_counts.get((lob, vintage), 0) for lob in nonkmx_lobs)
    rows.append(row)

    row = {'Flag': 'KMX_RECORDS', 'LOB_Applicability': 'KMX-only'}
    for vintage in vintages:
        row[vintage] = record_counts.get(('KMX', vintage), 0)
    rows.append(row)

    row = {'Flag': 'POS_TOTAL_RECORDS', 'LOB_Applicability': 'POS'}
    for vintage in vintages:
        row[vintage] = sum(record_counts.get((lob, vintage), 0) for lob in POS_LOBS)
    rows.append(row)

    row = {'Flag': 'NONKMX_AMT_FINANCED', 'LOB_Applicability': 'NonKMX-only'}
    for vintage in vintages:
        row[vintage] = sum(af_weights.get((lob, vintage), 0) for lob in nonkmx_lobs)
    rows.append(row)

    row = {'Flag': 'KMX_AMT_FINANCED', 'LOB_Applicability': 'KMX-only'}
    for vintage in vintages:
        row[vintage] = af_weights.get(('KMX', vintage), 0)
    rows.append(row)

    row = {'Flag': 'POS_TOTAL_AMT_FINANCED', 'LOB_Applicability': 'POS'}
    for vintage in vintages:
        row[vintage] = sum(af_weights.get((lob, vintage), 0) for lob in POS_LOBS)
    rows.append(row)

    return pd.DataFrame(rows)


pos_flags_df = build_pos_flag_means(
    flag_results_by_lob, af_by_lob, all_vintages, NONKMX_LOBS, record_counts_by_lob
)

print("POS Flag Means:")
display(pos_flags_df)

POS Flag Means:


,Flag,LOB_Applicability,2023 Q1,2023 Q2,2023 Q3,2023 Q4,2024 Q1,2024 Q2,2024 Q3,2024 Q4,2025 Q1,2025 Q2,2025 Q3,2025 Q4,2026 Q1,2026 Q2,2026 Q3
0,--- POS-WIDE FLAGS ---,,,,,,,,,,,,,,,,
1,student_loan_flag,POS (all LOBs),0.04384,0.048857,0.023328,0.030105,0.244302,0.22213,0.222121,0.256063,0.270763,0.21641,0.221508,0.174731,0.190582,0.177295,0.156557
2,fraud_adjustment_mean,POS (all LOBs),1.004008,1.005317,1.004398,1.00414,0.995068,0.994204,0.9964,0.996221,0.996592,0.993918,0.991418,0.995724,0.99333,1.002705,1.00204
3,driver_flag,POS (all LOBs),0.01354,0.015266,0.013399,0.013414,0.011479,0.014554,0.016442,0.019328,0.016146,0.019421,0.018,0.017629,0.014253,0.015972,0.012424
4,seasonal_employment_flag,POS (all LOBs),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.008768,0.010093,0.010441,0.010935,0.009612,0.01224,0.008031
5,toyho_flag,POS (all LOBs),0.123212,0.134258,0.135244,0.150439,0.142653,0.157237,0.164231,0.247364,0.271726,0.275391,0.285312,0.31105,0.242197,0.23369,0.217037
6,Chime (POS-wide),POS (all LOBs),0.158907,0.140575,0.133783,0.130586,0.184916,0.154092,0.160328,0.171739,0.239589,0.147158,0.178356,0.161517,0.253599,0.232258,0.200273
7,,,,,,,,,,,,,,,,,
8,--- NONKMX-ONLY FLAGS ---,,,,,,,,,,,,,,,,
9,prev_co_flag,NonKMX-only,0.002104,0.001476,0.001609,0.001635,0.001968,0.002003,0.00253,0.002503,0.003025,0.003963,0.002708,0.002867,0.005888,0.00713,0.006734


In [10]:
# =============================================================================
# CELL 10: EXCEL EXPORT
# =============================================================================

with pd.ExcelWriter(EXCEL_OUTPUT, engine='openpyxl') as writer:
    pos_attribution_df.to_excel(writer, sheet_name='POS Attribution')
    pos_flags_df.to_excel(writer, sheet_name='POS Flag Means', index=False)

    for lob in POS_LOBS:
        lob_data = {}
        for vintage in all_vintages:
            key = (lob, vintage)
            if key not in attribution_by_lob:
                continue
            attr = attribution_by_lob[key]
            pw = pop_weighting_by_lob.get(key, 0.0)
            gli = ragu_gli_by_lob.get(key, 0.0)
            col = {**attr, '--- Population Weighting': pw, '--- TOTAL (GLI)': gli}
            lob_data[vintage] = col

        if lob_data:
            all_labels = []
            for col in lob_data.values():
                for k in col.keys():
                    if k not in all_labels:
                        all_labels.append(k)
            lob_df = pd.DataFrame(
                {v: {r: lob_data[v].get(r, 0.0) for r in all_labels} for v in sorted(lob_data.keys())},
                index=all_labels
            )
            lob_df.to_excel(writer, sheet_name=f'{lob} Attribution')

print(f"Exported to: {EXCEL_OUTPUT}")
print("[PROGRESS] Export Complete")

Exported to: pos_gl_attribution.xlsx
[PROGRESS] Export Complete
